In [ ]:
import json

In [ ]:
data_dir='./dataset/gss/gss_grouped.json' 

In [ ]:
with open(data_dir) as f:
    data=json.load(f)

In [ ]:
data_dir='./dataset/gss/gss_per_user_with_gt.json' 

In [ ]:
with open(data_dir) as f:
    gss=json.load(f)

In [ ]:
len(gss)

In [ ]:
import random
from copy import deepcopy


def split_by_individuals(data, portions=[0.2, 0.4, 0.6, 0.8], random_seed=1004):
    """
    Split data by randomly selecting cumulative portions of individuals.
    
    The individuals excluded at 20% are a subset of those excluded at 40%, etc.
    
    Args:
        data: List of instances with 'id', 'feature', 'responses'
        portions: List of floats (must be sorted ascending)
        random_seed: Random seed for reproducibility
    
    Returns:
        Dictionary mapping portion keys to split results
    """
    rng = random.Random(random_seed)
    
    # Get all individual IDs and shuffle once
    all_ids = [instance['id'] for instance in data]
    shuffled_ids = all_ids[:]
    rng.shuffle(shuffled_ids)
    
    results = {}
    
    for portion in portions:
        portion_key = f"{int(portion * 100)}"
        
        # Take the first `num_to_select` from the SAME shuffled list
        # This guarantees cumulative inclusion
        num_to_select = int(len(shuffled_ids) * portion)
        selected_ids = set(shuffled_ids[:num_to_select])
        
        subgroup1 = [inst for inst in data if inst['id'] in selected_ids]
        subgroup2 = [inst for inst in data if inst['id'] not in selected_ids]
        
        results[portion_key] = {
            'subgroup1': subgroup1,
            'subgroup2': subgroup2,
            'selected_ids': selected_ids
        }
    
    return results


def split_by_questions(data, portions=[0.2, 0.4, 0.6, 0.8], random_seed=1004):
    """
    Split data by randomly selecting cumulative portions of questions.
    
    The questions excluded at 20% are a subset of those excluded at 40%, etc.
    
    Args:
        data: List of instances with 'id', 'feature', 'responses'
        portions: List of floats (must be sorted ascending)
        random_seed: Random seed for reproducibility
    
    Returns:
        Dictionary mapping portion keys to split results
    """
    rng = random.Random(random_seed)
    
    # Get all unique question variables
    all_questions = set()
    for instance in data:
        for response in instance['responses']:
            all_questions.add(response['Question_var'])
    
    # Shuffle once — this defines the cumulative ordering
    shuffled_questions = list(all_questions)
    rng.shuffle(shuffled_questions)
    
    results = {}
    
    for portion in portions:
        portion_key = f"{int(portion * 100)}"
        
        # Take the first `num_to_select` from the SAME shuffled list
        # 20% → first 20%, 40% → first 40% (which INCLUDES the 20%), etc.
        num_to_select = int(len(shuffled_questions) * portion)
        selected_questions = set(shuffled_questions[:num_to_select])
        
        subgroup1 = []
        subgroup2 = []
        
        for instance in data:
            instance1 = {
                'id': instance['id'],
                'feature': deepcopy(instance['feature']),
                'responses': [r for r in instance['responses']
                             if r['Question_var'] in selected_questions]
            }
            instance2 = {
                'id': instance['id'],
                'feature': deepcopy(instance['feature']),
                'responses': [r for r in instance['responses']
                             if r['Question_var'] not in selected_questions]
            }
            
            subgroup1.append(instance1)
            subgroup2.append(instance2)
        
        results[portion_key] = {
            'subgroup1': subgroup1,
            'subgroup2': subgroup2,
            'selected_questions': selected_questions
        }
    
    return results


def perform_all_splits(data, portions=[0.2, 0.4, 0.6, 0.8], random_seed=1004):
    """
    Perform all cumulative splits for given portions.
    
    Returns:
        Dictionary containing all splits
    """
    # Ensure portions are sorted for cumulative logic to make sense
    portions = sorted(portions)
    
    results = {
        'by_individuals': split_by_individuals(data, portions, random_seed),
        'by_questions': split_by_questions(data, portions, random_seed)
    }
    
    return results

In [ ]:
results=perform_all_splits(gss, portions=[0.2, 0.4, 0.6, 0.8], random_seed=1000)

In [ ]:
results['by_individuals'].keys()

In [ ]:
len(results['by_questions']['20']['subgroup1'])

In [ ]:
def verify_cumulative_splits(results, split_type='by_questions'):
    """Verify that splits are properly cumulative."""
    portions = sorted(results[split_type].keys(), key=int)
    
    key = 'selected_questions' if split_type == 'by_questions' else 'selected_ids'
    
    for i in range(1, len(portions)):
        prev = results[split_type][portions[i-1]][key]
        curr = results[split_type][portions[i]][key]
        
        is_subset = prev.issubset(curr)
        newly_added = len(curr - prev)
        
        print(f"{portions[i-1]}% → {portions[i]}%: "
              f"subset={is_subset}, "
              f"prev={len(prev)}, curr={len(curr)}, "
              f"newly added={newly_added}")


In [ ]:
verify_cumulative_splits(results)

In [ ]:
random_index=4

In [ ]:
for percent in ['20', '40', '60', '80']:
    persona_ind=results['by_individuals'][percent]['subgroup1']
    key_ind=list(results['by_individuals'][percent]['selected_ids'])

    with open(f'./dataset/gss/split/individual/gss_{percent}_ind{random_index}_per_user_with_gt.json', 'w') as f:
        json.dump(persona_ind, f)
    with open(f'./dataset/gss/split/individual/gss_{percent}_ind{random_index}_selected_key.json', 'w') as f:
        json.dump(key_ind, f)
    
    persona_q=results['by_questions'][percent]['subgroup1']
    key_q=list(results['by_questions'][percent]['selected_questions'])
    with open(f'./dataset/gss/split/question/gss_{percent}_q{random_index}_per_user_with_gt.json', 'w') as f:
        json.dump(persona_q, f)
    with open(f'./dataset/gss/split/question/gss_{percent}_q{random_index}_selected_key.json', 'w') as f:
        json.dump(key_q, f)


### Test set split - Question

In [ ]:
for percent in ['20', '40', '60', '80']:
    with open(f'./dataset/gss/gss_grouped.json') as f:
        data=json.load(f)
    with open(f'./dataset/gss/gss_per_user_with_gt.json') as f:
        beh=json.load(f)    
    with open(f'./dataset/gss/split/question/gss_{percent}_q{random_index}_selected_key.json') as f:
        keys_to_exclude=json.load(f)
    with open(f'./dataset/gss/split/question/gss_{percent}_q{random_index}_per_user_with_gt.json') as f:
        check=json.load(f)
    data_wo_q=[]
    question_idx2_key={}
    for idx, res in enumerate(beh[0]['responses']):
        question_idx2_key[idx]=res['Question_var']

    for d in data:
        q_var=question_idx2_key[d['question_idx']]
        if q_var not in keys_to_exclude:
            data_wo_q.append(d)
    
    for res in check[0]['responses']:
        assert(res['Question_var'] in keys_to_exclude)
        
    with open(f'./dataset/gss/split/question/gss_{percent}_q{random_index}_grouped.json','w') as f:
        json.dump(data_wo_q, f)
    

### Test set split - Individual

In [ ]:
for percent in ['20', '40', '60', '80']:
    with open(f'./dataset/gss/gss_per_user_with_gt.json') as f:
        beh=json.load(f)    
    with open(f'./dataset/gss/split/individual/gss_{percent}_ind{random_index}_selected_key.json') as f:
        keys_to_exclude=json.load(f)
    with open(f'./dataset/gss/split/individual/gss_{percent}_ind{random_index}_per_user_with_gt.json') as f:
        check=json.load(f)

    data_wo_ind=[]
    for d in beh:
        if d['id'] not in keys_to_exclude:
            data_wo_ind.append(d)
    with open(f'./dataset/gss/split/individual/gss_{percent}_ind{random_index}_per_user_with_gt_test.json', 'w') as f:
        json.dump(data_wo_ind, f)    

### Group test set

In [ ]:
from collections import defaultdict

In [ ]:
def map_instance_features(instance):
    """Map individual instance features to reference format, keeping only specified features."""
    original_features = instance['feature']
    responses = {r['Question_var']: r['Response'] for r in instance['responses']}
    
    # Initialize mapped features dict (only for specified features)
    mapped_features = {}
    
    # AGE mapping (numeric to range)
    age_str = original_features.get('age', '')
    if age_str:
        if age_str == '89 or older':
            mapped_features['AGE'] = '65+'
        else:
            try:
                age = float(age_str)
                if 18 <= age <= 29:
                    mapped_features['AGE'] = '18-29'
                elif 30 <= age <= 49:
                    mapped_features['AGE'] = '30-49'
                elif 50 <= age <= 64:
                    mapped_features['AGE'] = '50-64'
                elif age >= 65:
                    mapped_features['AGE'] = '65+'
                else:
                    mapped_features['AGE'] = ''
            except:
                mapped_features['AGE'] = ''
    else:
        mapped_features['AGE'] = ''
    
    # EDUCATION mapping (from 'degree')
    degree = original_features.get('degree', '')
    education_map = {
        "bachelor's": "College graduate/some postgrad",
        "high school": "High school graduate",
        "associate/junior college": "Associate's degree",
        "less than high school": "Less than high school",
        "graduate": "Postgraduate"
    }
    mapped_features['EDUCATION'] = education_map.get(degree, '')
    
    # INCOME mapping
    income = original_features.get('income', '')
    if income == '':
        mapped_features['INCOME'] = ''
    elif income == '$25,000 or more':
        mapped_features['INCOME'] = '$30,000-$50,000'
    elif income in ['$20,000 to $24,999', '$15,000 to $19,999', '$10,000 to $14,999',
                    '$8,000 to $9,999', '$7,000 to $7,999', '$6,000 to $6,999',
                    '$5,000 to $5,999', '$4,000 to $4,999', '$3,000 to $3,999',
                    '$1,000 to $2,999', 'under $1,000']:
        mapped_features['INCOME'] = 'Less than $30,000'
    else:
        mapped_features['INCOME'] = ''
    
    # MARITAL mapping
    marital = original_features.get('marital', '')
    marital_map = {
        'married': 'Married',
        'separated': 'Separated',
        'widowed': 'Widowed',
        'divorced': 'Divorced',
        'never married': 'Never been married'
    }
    mapped_features['MARITAL'] = marital_map.get(marital, '')
    
    # RACE mapping (combining race and hispanic)
    race = original_features.get('race', '')
    hispanic = original_features.get('hispanic', '')
    
    if hispanic and hispanic != 'not hispanic' and hispanic != '':
        mapped_features['RACE'] = 'Hispanic'
    else:
        race_map = {'white': 'White', 'black': 'Black', 'other': 'Other'}
        mapped_features['RACE'] = race_map.get(race, '')
    
    # CREGION mapping (from 'region')
    region = original_features.get('region', '')
    region_map = {'south': 'South', 'west': 'West', 'midwest': 'Midwest', 'northeast': 'Northeast'}
    mapped_features['CREGION'] = region_map.get(region, '')
    
    # RELIG mapping
    relig = original_features.get('relig', '')
    relig_map = {
        'protestant': 'Protestant',
        'catholic': 'Roman Catholic',
        'jewish': 'Jewish',
        'muslim/islam': 'Muslim',
        'buddhism': 'Buddhist',
        'hinduism': 'Hindu',
        'christian': 'Christian',
        'orthodox-christian': 'Orthodox',
        'none': 'Nothing in particular',
        'other': 'Other',
        'inter-nondenominational': 'Other',
        'other eastern religions': 'Other',
        'native american': 'Other'
    }
    mapped_features['RELIG'] = relig_map.get(relig, '')
    
    # SEX mapping
    sex = original_features.get('sex', '')
    sex_map = {'male': 'Male', 'female': 'Female'}
    mapped_features['SEX'] = sex_map.get(sex, '')
    
    # RELIGATTEND mapping (from 'attend' response)
    attend = responses.get('attend', '')
    attend_map = {
        'never': 'Never',
        'less than once a year': 'Seldom',
        'once a year': 'A few times a year',
        'several times a year': 'A few times a year',
        'once a month': 'Once or twice a month',
        '2-3 times a month': 'Once or twice a month',
        'nearly every week': 'Once a week',
        'every week': 'Once a week',
        'more than once a week': 'More than once a week'
    }
    mapped_features['RELIGATTEND'] = attend_map.get(attend, '')
    
    # POLPARTY mapping (from 'partyid' response)
    partyid = responses.get('partyid', '')
    if partyid:
        if 'democrat' in partyid.lower():
            mapped_features['POLPARTY'] = 'Democrat'
        elif 'republican' in partyid.lower():
            mapped_features['POLPARTY'] = 'Republican'
        elif 'independent' in partyid.lower():
            mapped_features['POLPARTY'] = 'Independent'
        else:
            mapped_features['POLPARTY'] = 'Other'
    else:
        mapped_features['POLPARTY'] = ''
    
    # POLIDEOLOGY mapping (from 'polviews' response)
    polviews = responses.get('polviews', '')
    if polviews:
        polviews_lower = polviews.lower()
        if 'extremely liberal' in polviews_lower:
            mapped_features['POLIDEOLOGY'] = 'Very liberal'
        elif 'extremely conservative' in polviews_lower:
            mapped_features['POLIDEOLOGY'] = 'Very conservative'
        elif 'slightly liberal' in polviews_lower or polviews_lower == 'liberal':
            mapped_features['POLIDEOLOGY'] = 'Liberal'
        elif 'slightly conservative' in polviews_lower or polviews_lower == 'conservative':
            mapped_features['POLIDEOLOGY'] = 'Conservative'
        elif 'moderate' in polviews_lower:
            mapped_features['POLIDEOLOGY'] = 'Moderate'
        else:
            mapped_features['POLIDEOLOGY'] = ''
    else:
        mapped_features['POLIDEOLOGY'] = ''
    
    return {
        'id': instance['id'],
        'feature': mapped_features,
        'responses': instance['responses']
    }


def process_gss(gss):
    """
    Process list of GSS instances and map features to reference format.
    Only keeps specified demographic features.
    
    Args:
        gss: List of instance dictionaries with 'id', 'feature', and 'responses'
        
    Returns:
        processed: List of processed instances with mapped features
    """
    processed = []
    
    for instance in gss:
        try:
            mapped_instance = map_instance_features(instance)
            processed.append(mapped_instance)
        except Exception as e:
            print(f"Error processing instance {instance.get('id', 'unknown')}: {e}")
            continue
    
    return processed

import json
from collections import defaultdict
import string
from typing import List, Dict, Any, Optional

# Define all possible demographic values (only specified features)
DEMOGRAPHIC_VALUES = {
    'CREGION': ['Midwest', 'Northeast', 'South', 'West'],
    'AGE': ['18-29', '30-49', '50-64', '65+'],
    'SEX': ['Female', 'Male', 'In some other way'],
    'EDUCATION': ['Less than high school', 
                  'High school graduate', 
                  'Some college, no degree', 
                  "Associate's degree", 
                  'College graduate/some postgrad', 
                  'Postgraduate'],
    'MARITAL': ['Married', 
                'Living with a partner', 
                'Divorced', 
                'Separated', 
                'Widowed', 
                'Never been married'],
    'RELIG': ['Protestant', 
              'Roman Catholic', 
              'Mormon', 
              'Orthodox', 
              'Jewish', 
              'Muslim', 
              'Buddhist', 
              'Hindu', 
              'Atheist', 
              'Agnostic', 
              'Nothing in particular', 
              'Christian', 
              'Unitarian', 
              'Other'],
    'RELIGATTEND': ['More than once a week', 
                    'Once a week', 
                    'Once or twice a month', 
                    'A few times a year', 
                    'Seldom', 
                    'Never'],
    'POLPARTY': ['Democrat', 'Republican', 'Independent', 'Other'],
    'POLIDEOLOGY': ['Very liberal', 'Liberal', 'Moderate', 'Conservative', 'Very conservative'],
    'INCOME': ['Less than $30,000', 
               '$30,000-$50,000', 
               '$50,000-$75,000', 
               '$75,000-$100,000', 
               '$100,000 or more'],
    'RACE': ['White', 'Black', 'Asian', 'Hispanic', 'Mixed Race', 'Other']
}

# Define demographic description templates
DEMOGRAPHIC_DESCRIPTIONS = {
    'CREGION': {
        'Midwest': 'You are from the Midwest region.',
        'Northeast': 'You are from the Northeast region.',
        'South': 'You are from the South region.',
        'West': 'You are from the West region.'
    },
    'AGE': {
        '18-29': 'You are aged 18-29.',
        '30-49': 'You are aged 30-49.',
        '50-64': 'You are aged 50-64.',
        '65+': 'You are aged 65 or older.'
    },
    'SEX': {
        'Female': 'You are Female.',
        'Male': 'You are Male.',
        'In some other way': 'You identify your gender in some other way.'
    },
    'EDUCATION': {
        'Less than high school': 'Your highest education level is less than high school.',
        'High school graduate': 'Your highest education level is high school graduate.',
        'Some college, no degree': 'Your highest education level is some college, no degree.',
        "Associate's degree": "Your highest education level is an Associate's degree.",
        'College graduate/some postgrad': 'Your highest education level is college graduate or some postgraduate.',
        'Postgraduate': 'Your highest education level is postgraduate.'
    },
    'MARITAL': {
        'Married': 'You are married.',
        'Living with a partner': 'You are living with a partner.',
        'Widowed': 'You are widowed.',
        'Divorced': 'You are divorced.',
        'Separated': 'You are separated.',
        'Never been married': 'You have never been married.'
    },
    'RELIG': {
        'Protestant': 'Your religion is Protestant.',
        'Roman Catholic': 'Your religion is Roman Catholic.',
        'Mormon': 'Your religion is Mormon.',
        'Orthodox': 'Your religion is Orthodox Christian.',
        'Jewish': 'Your religion is Jewish.',
        'Muslim': 'Your religion is Muslim.',
        'Buddhist': 'Your religion is Buddhist.',
        'Hindu': 'Your religion is Hindu.',
        'Atheist': 'You are an Atheist.',
        'Agnostic': 'You are Agnostic.',
        'Nothing in particular': 'You have no particular religion.',
        'Unitarian': 'Your religion is Unitarian.',
        'Christian': 'Your religion is Christian.',
        'Other': 'Your religion is other.'
    },
    'RELIGATTEND': {
        'More than once a week': 'You attend religious services more than once a week.',
        'Once a week': 'You attend religious services once a week.',
        'Once or twice a month': 'You attend religious services once or twice a month.',
        'A few times a year': 'You attend religious services a few times a year.',
        'Seldom': 'You seldom attend religious services.',
        'Never': 'You never attend religious services.'
    },
    'POLPARTY': {
        'Republican': 'You identify as a Republican.',
        'Democrat': 'You identify as a Democrat.',
        'Independent': 'You identify as an Independent.',
        'Other': 'You identify with another political party.'
    },
    'INCOME': {
        'Less than $30,000': 'Your household income is less than $30,000.',
        '$30,000-$50,000': 'Your household income is $30,000-$50,000.',
        '$50,000-$75,000': 'Your household income is $50,000-$75,000.',
        '$75,000-$100,000': 'Your household income is $75,000-$100,000.',
        '$100,000 or more': 'Your household income is $100,000 or more.'
    },
    'POLIDEOLOGY': {
        'Very conservative': 'Your political ideology is very conservative.',
        'Conservative': 'Your political ideology is conservative.',
        'Moderate': 'Your political ideology is moderate.',
        'Liberal': 'Your political ideology is liberal.',
        'Very liberal': 'Your political ideology is very liberal.'
    },
    'RACE': {
        'White': 'Your race/ethnicity is White.',
        'Black': 'Your race/ethnicity is Black.',
        'Asian': 'Your race/ethnicity is Asian.',
        'Hispanic': 'Your race/ethnicity is Hispanic.',
        'Mixed Race': 'Your race/ethnicity is Mixed Race.',
        'Other': 'Your race/ethnicity is other.'
    }
}

# Values to skip (empty or refused)
SKIP_VALUES = {'', 'Refused'}

# Features to use for demographic grouping (only specified features)
DEMOGRAPHIC_KEYS = ['CREGION', 'AGE', 'SEX', 'EDUCATION', 'MARITAL', 
                    'RELIG', 'RELIGATTEND', 'POLPARTY', 'INCOME', 'POLIDEOLOGY', 'RACE']


def parse_possible_answers(possible_answers_str: str) -> List[str]:
    """
    Parse possible answers from string format.
    Handles both semicolon-separated strings and list representations.
    
    Args:
        possible_answers_str: String containing possible answers
        
    Returns:
        List of possible answers
    """
    if isinstance(possible_answers_str, list):
        return possible_answers_str
    
    # Check if it's a string representation of a list
    if possible_answers_str.strip().startswith('['):
        return eval(possible_answers_str)
    
    # Semicolon-separated format
    return [ans.strip() for ans in possible_answers_str.split(';')]


def normalize_response(response: str, possible_answers: List[str]) -> str:
    """
    Normalize a response to match one of the possible answers (case-insensitive).
    
    Args:
        response: The response string
        possible_answers: List of possible answers
        
    Returns:
        Matched possible answer or empty string if no match
    """
    if not response:
        return ''
    
    response_lower = response.lower().strip()
    
    # Create a mapping of lowercase answers to original answers
    answer_map = {ans.lower().strip(): ans for ans in possible_answers}
    
    # Direct match
    if response_lower in answer_map:
        return answer_map[response_lower]
    
    # Partial match for common variations
    for ans_lower, ans_original in answer_map.items():
        # Check if response is contained in answer or vice versa
        if response_lower in ans_lower or ans_lower in response_lower:
            return ans_original
    
    # No match found - return original response (will be counted if it matches exactly)
    return response


def preprocess_survey_data(
    data_list: List[Dict], 
    wave: float, 
    year: int = 2017,
    min_group_size: int = 0,
    min_group_size_overall: Optional[int] = None,
    dataset_name: str = 'OpinionQA'
) -> List[Dict]:
    """
    Preprocess survey data to create grouped responses.
    
    Args:
        data_list: List of survey instances, each with 'id', 'feature', and 'responses'
        wave: Wave number for the survey
        year: Survey year
        min_group_size: Minimum group size threshold for demographic groups (num_grouping_vars=1).
                        Groups with size below this threshold will be filtered out.
        min_group_size_overall: Minimum group size threshold for overall groups (num_grouping_vars=0).
                                If None, no filtering is applied to overall groups.
        dataset_name: Name of the dataset
    
    Returns:
        List of processed instances with grouped distributions
    """
    
    # Step 1: Extract all unique questions and their metadata
    questions = {}
    for instance in data_list:
        for response in instance['responses']:
            q_var = response['Question_var']
            if q_var not in questions:
                # Parse possible answers
                possible_answers = parse_possible_answers(response['Possible_Answers'])
                
                questions[q_var] = {
                    'question': response['Question'],
                    'possible_answers': possible_answers,
                    'question_var': q_var
                }
    
    # Step 2: Collect all responses grouped by question variable
    question_responses = defaultdict(list)  # q_var -> list of {'response': str, 'features': dict}
    
    for instance in data_list:
        # Extract only demographic features
        features = {k: v for k, v in instance['feature'].items() if k in DEMOGRAPHIC_KEYS}
        
        for response in instance['responses']:
            q_var = response['Question_var']
            resp_value = response['Response']
            
            # Normalize the response to match possible answers
            possible_answers = questions[q_var]['possible_answers']
            normalized_response = normalize_response(resp_value, possible_answers)
            
            question_responses[q_var].append({
                'response': normalized_response,
                'features': features
            })
    
    # Step 3: Helper function to create answer options mapping
    def create_answer_options(possible_answers: List[str]) -> Dict[str, str]:
        answer_options = {}
        letters = string.ascii_uppercase
        for i, ans in enumerate(possible_answers):
            if i < len(letters):
                answer_options[letters[i]] = ans
        return answer_options
    
    # Step 4: Helper function to create input template
    def create_input_template(question: str, answer_options: Dict[str, str]) -> str:
        input_template = f"{question}\n\nOptions:\n"
        for letter in sorted(answer_options.keys()):
            input_template += f"({letter}): {answer_options[letter]}\n"
        return input_template.strip()
    
    # Step 5: Helper function to calculate answer distribution
    def calculate_distribution(responses: List[Dict], answer_options: Dict[str, str]) -> tuple:
        answer_counts = defaultdict(int)
        valid_responses = [r for r in responses if r['response']]  # Only count non-empty responses
        total = len(valid_responses)
        
        for r in valid_responses:
            answer_counts[r['response']] += 1
        
        human_answer = {}
        for letter, ans in answer_options.items():
            count = answer_counts.get(ans, 0)
            human_answer[letter] = (count / total * 100) if total > 0 else 0.0
        
        return human_answer, total
    
    # Step 6: Helper function to get demographic description
    def get_demographic_description(demo_key: str, demo_value: str) -> str:
        if demo_key in DEMOGRAPHIC_DESCRIPTIONS:
            if demo_value in DEMOGRAPHIC_DESCRIPTIONS[demo_key]:
                return DEMOGRAPHIC_DESCRIPTIONS[demo_key][demo_value]
        # Fallback for values not in descriptions
        return f'Your {demo_key} is {demo_value}.'
    
    # Step 7: Create output list
    output = []
    
    # Track filtered groups for reporting
    filtered_overall = 0
    filtered_demographic = 0
    
    # Get ordered list of question variables
    question_vars = list(questions.keys())
    
    # ==========================================
    # FIRST N INSTANCES: Overall distributions (num_grouping_vars = 0)
    # ==========================================
    for question_idx, q_var in enumerate(question_vars):
        q_info = questions[q_var]
        responses_list = question_responses[q_var]
        
        # Create answer options
        answer_options = create_answer_options(q_info['possible_answers'])
        
        # Create input template
        input_template = create_input_template(q_info['question'], answer_options)
        
        # Calculate overall distribution
        human_answer, group_size = calculate_distribution(responses_list, answer_options)
        
        # Skip questions with no valid responses
        if group_size == 0:
            continue
        
        # Apply threshold for overall groups if specified
        if min_group_size_overall is not None and group_size < min_group_size_overall:
            filtered_overall += 1
            continue
        
        output.append({
            'group_prompt_template': f'You are from the United States. The year is {year}.',
            'group_prompt_variable_map': {},
            'input_template': input_template,
            'input_variable_map': {},
            'output_format': 'choice',
            'answer_options': answer_options,
            'auxiliary': {},
            'group_size': group_size,
            'human_answer': human_answer,
            'wave': wave,
            'num_grouping_vars': 0,
            'dataset_name': dataset_name,
            'question_idx': question_idx
        })
    
    # ==========================================
    # AFTER N: Grouped by demographic features (num_grouping_vars = 1)
    # ==========================================
    for question_idx, q_var in enumerate(question_vars):
        q_info = questions[q_var]
        responses_list = question_responses[q_var]
        
        # Create answer options
        answer_options = create_answer_options(q_info['possible_answers'])
        
        # Create input template
        input_template = create_input_template(q_info['question'], answer_options)
        
        # Group by each demographic feature
        for demo_key in DEMOGRAPHIC_KEYS:
            # Get unique values for this demographic from the data
            demo_values = set(r['features'].get(demo_key) for r in responses_list if r['features'].get(demo_key) is not None)
            
            for demo_value in sorted(demo_values):
                # Skip empty values and 'Refused'
                if demo_value in SKIP_VALUES:
                    continue
                
                # Filter responses for this demographic value
                filtered_responses = [
                    r for r in responses_list 
                    if r['features'].get(demo_key) == demo_value
                ]
                
                if not filtered_responses:
                    continue
                
                # Calculate distribution for this demographic group
                human_answer, group_size = calculate_distribution(filtered_responses, answer_options)
                
                # Skip groups with no valid responses
                if group_size == 0:
                    continue
                
                # Apply minimum group size threshold
                if group_size < min_group_size:
                    filtered_demographic += 1
                    continue
                
                # Get demographic description
                demo_description = get_demographic_description(demo_key, demo_value)
                
                # Create group prompt
                group_prompt = f'You are from the United States. The year is {year}. {demo_description}'
                
                output.append({
                    'group_prompt_template': group_prompt,
                    'group_prompt_variable_map': {demo_key: demo_value},
                    'input_template': input_template,
                    'input_variable_map': {},
                    'output_format': 'choice',
                    'answer_options': answer_options,
                    'auxiliary': {},
                    'group_size': group_size,
                    'human_answer': human_answer,
                    'wave': wave,
                    'num_grouping_vars': 1,
                    'dataset_name': dataset_name,
                    'question_idx': question_idx
                })
    
    # Print filtering summary
    print(f"Filtering Summary:")
    print(f"  - Total unique questions: {len(question_vars)}")
    print(f"  - Minimum group size for demographic groups: {min_group_size}")
    print(f"  - Minimum group size for overall groups: {min_group_size_overall if min_group_size_overall is not None else 'No threshold'}")
    print(f"  - Filtered out {filtered_overall} overall instances")
    print(f"  - Filtered out {filtered_demographic} demographic group instances")
    print(f"  - Total instances in output: {len(output)}")
    
    return output


def load_and_preprocess(
    input_file: str, 
    output_file: str, 
    wave: float, 
    year: int = 2017,
    min_group_size: int = 0,
    min_group_size_overall: Optional[int] = None,
    dataset_name: str = 'OpinionQA'
):
    """
    Load data from file, preprocess, and save results.
    
    Args:
        input_file: Path to input JSON file containing list of survey instances
        output_file: Path to output JSON file
        wave: Wave number for the survey
        year: Survey year
        min_group_size: Minimum group size threshold for demographic groups
        min_group_size_overall: Minimum group size threshold for overall groups
        dataset_name: Name of the dataset
    """
    # Load data
    with open(input_file, 'r') as f:
        data_list = json.load(f)
    
    # Preprocess
    processed_data = preprocess_survey_data(
        data_list, 
        wave=wave, 
        year=year,
        min_group_size=min_group_size,
        min_group_size_overall=min_group_size_overall,
        dataset_name=dataset_name
    )
    
    # Save results
    with open(output_file, 'w') as f:
        json.dump(processed_data, f, indent=2)
    
    print(f"Output saved to {output_file}")
    
    return processed_data


def get_response_mapping_report(data_list: List[Dict]) -> Dict:
    """
    Generate a report showing how responses are mapped to possible answers.
    Useful for debugging response normalization.
    
    Args:
        data_list: List of survey instances
        
    Returns:
        Dictionary with mapping information per question
    """
    report = {}
    
    for instance in data_list:
        for response in instance['responses']:
            q_var = response['Question_var']
            if q_var not in report:
                possible_answers = parse_possible_answers(response['Possible_Answers'])
                report[q_var] = {
                    'possible_answers': possible_answers,
                    'responses_found': defaultdict(int),
                    'unmapped_responses': defaultdict(int)
                }
            
            resp_value = response['Response']
            if resp_value:
                normalized = normalize_response(resp_value, report[q_var]['possible_answers'])
                if normalized in report[q_var]['possible_answers']:
                    report[q_var]['responses_found'][normalized] += 1
                else:
                    report[q_var]['unmapped_responses'][resp_value] += 1
    
    # Convert defaultdicts to regular dicts
    for q_var in report:
        report[q_var]['responses_found'] = dict(report[q_var]['responses_found'])
        report[q_var]['unmapped_responses'] = dict(report[q_var]['unmapped_responses'])
    
    return report


def get_demographic_summary(data_list: List[Dict]) -> Dict[str, Dict[str, int]]:
    """Get count of each demographic value in the data (only for grouping demographics)."""
    demo_counts = defaultdict(lambda: defaultdict(int))
    for instance in data_list:
        for key in DEMOGRAPHIC_KEYS:
            value = instance['feature'].get(key, '')
            demo_counts[key][value] += 1
    return {k: dict(v) for k, v in demo_counts.items()}


def validate_demographics(data_list: List[Dict]) -> Dict[str, List[str]]:
    """Check for any demographic values not in the predefined list."""
    invalid_values = defaultdict(list)
    for instance in data_list:
        for key in DEMOGRAPHIC_KEYS:
            value = instance['feature'].get(key)
            if value is not None and key in DEMOGRAPHIC_VALUES:
                if value not in DEMOGRAPHIC_VALUES[key]:
                    if value not in invalid_values[key]:
                        invalid_values[key].append(value)
    return dict(invalid_values)


In [ ]:
for percent in ['20', '40', '60', '80']:
    with open(f'./dataset/gss/split/individual/gss_{percent}_ind{random_index}_per_user_with_gt_test.json') as f:
        gss=json.load(f)
    gss_f=defaultdict(list)
    for d in gss:
        for k,v in d['feature'].items():
            gss_f[k].append(v)
    for k,v in gss_f.items():
        gss_f[k]=list(set(gss_f[k]))
    processed_gss = process_gss(gss)
    processed = preprocess_survey_data(
        processed_gss, 
        wave=0.0, 
        year=2024, 
        min_group_size=100,
        dataset_name='GSS'
    )
    with open(f'./dataset/gss/split/individual/gss_{percent}_ind{random_index}_grouped.json','w') as f:
        json.dump(processed, f)
    print(len(processed))

In [ ]:
beh[0]

In [ ]:
data[0]

In [ ]:
question_idx2_key={}
for idx, res in enumerate(beh[0]['responses']):
    question_idx2_key[idx]=res['Question_var']

In [ ]:
data[0]

In [ ]:
lis=[]
for res in data[0]['responses']:
    lis.append(res['Question_var'])

In [ ]:
keys[0]

In [ ]:
type(key_ind)

In [ ]:
results['by_questions'][percent].keys()

In [ ]:
len(results['by_individuals']['20%']['selected_ids'])

In [ ]:
len(results['by_individuals']['10%']['subgroup2'])

In [ ]:
len(results['by_questions']['10%']['subgroup1'][0]['responses'])

In [ ]:
len(results['by_questions']['10%']['subgroup2'][0]['responses'])

In [ ]:
len(gss[0]['responses'])